<a href="https://colab.research.google.com/github/PreethamHD/DP-MMFL/blob/main/notebooks/12_train_multimodal_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
from pathlib import Path
from google.colab import drive, userdata

# 1. Mount persistent storage (Google Drive)
if not os.path.exists("/content/drive"):
    drive.mount("/content/drive")

# 2. Configure Git remote access
GITHUB_USERNAME = "PreethamHD"
REPO_NAME = "DP-MMFL"
REPO_DIR = Path(f"/content/{REPO_NAME}")
SRC_DIR = REPO_DIR / "src"

!git config --global user.name "PreethamHD"
!git config --global user.email "your_email@example.com"

try:
    token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{token}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
except Exception:
    repo_url = f"https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"

# 3. Clone or pull repo
%cd /content
if not REPO_DIR.exists():
    !git clone {repo_url}
    %cd {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git pull origin main

# 4. Inject module search path
src_str = str(SRC_DIR.resolve())
if src_str not in sys.path:
    sys.path.insert(0, src_str)

# 5. Install dependencies
!pip install -q transformers pyarrow scikit-learn

# 6. Verify Paths, Manifest, Seed & CUDA
import torch
from dp_mmfl.utils.seed import set_seed

SEED = 42
set_seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data" / "processed" / "chexpert_plus_manifest.parquet"
CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
E1_METRICS_DIR = PROJECT_ROOT / "results" / "E1_image_baseline" / "metrics"

print("=" * 60)
print("PROJECT_ROOT:       ", PROJECT_ROOT)
print("MANIFEST_PATH:      ", MANIFEST_PATH)
print("Manifest Exists:    ", MANIFEST_PATH.exists())
print("256 Cache Exists:   ", CACHE_ROOT_256.exists())
print("E1 Split File:      ", (E1_METRICS_DIR / "split_info.json").exists())
print("CUDA Available:     ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:           ", torch.cuda.get_device_name(0))
    print("VRAM Allocated:     ", f"{torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")
print("=" * 60)

Mounted at /content/drive
/content
Cloning into 'DP-MMFL'...
remote: Enumerating objects: 154, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (102/102), done.
remote: Total 154 (delta 65), reused 99 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (154/154), 518.80 KiB | 2.11 MiB/s, done.
Resolving deltas: 100% (65/65), done.
/content/DP-MMFL
PROJECT_ROOT:        /content/drive/MyDrive/DP-MMFL
MANIFEST_PATH:       /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest Exists:     True
256 Cache Exists:    False
E1 Split File:       True
CUDA Available:      True
GPU Name:            Tesla T4
VRAM Allocated:      0.00 MB


In [ ]:
print("=" * 60)
print("PROJECT_ROOT:       ", PROJECT_ROOT)
print("MANIFEST_PATH:      ", MANIFEST_PATH)
print("Manifest Exists:    ", MANIFEST_PATH.exists())
print("256 Cache Exists:   ", CACHE_ROOT_256.exists())
print("E1 Split File:      ", (E1_METRICS_DIR / "split_info.json").exists())
print("CUDA Available:     ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:           ", torch.cuda.get_device_name(0))
    print("VRAM Allocated:     ", f"{torch.cuda.memory_allocated(0) / (1024**2):.2f} MB")
print("=" * 60)

PROJECT_ROOT:        /content/drive/MyDrive/DP-MMFL
MANIFEST_PATH:       /content/drive/MyDrive/DP-MMFL/data/processed/chexpert_plus_manifest.parquet
Manifest Exists:     True
256 Cache Exists:    True
E1 Split File:       True
CUDA Available:      True
GPU Name:            Tesla T4
VRAM Allocated:      0.00 MB


In [ ]:
!pip install -q redivis transformers pyarrow scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.1/229.1 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.2/862.2 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.3/134.3 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 407.4/407.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 93.8 MB/s eta 0:00:00


In [ ]:
def evaluate_global(model, loader):
    model.eval()
    total_loss = 0.0
    total_valid_labels = 0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for batch in loader:
            images = batch["image"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True).float()
            masks = batch["label_mask"].to(device, non_blocking=True).float()

            ids = batch["sample_id"]
            if torch.is_tensor(ids) or isinstance(ids, np.ndarray):
                ids = ids.tolist()
            ids = [int(x) for x in ids]

            text_batch = torch.stack([embedding_by_id[sid] for sid in ids]).to(device, non_blocking=True)

            logits = model(images, text_batch)
            loss = masked_bce_with_logits(logits, labels, masks)

            batch_valid = masks.sum().item()
            total_loss += loss.item() * batch_valid
            total_valid_labels += batch_valid

            all_logits.append(logits.cpu())
            all_targets.append(labels.cpu())
            all_masks.append(masks.cpu())

    avg_loss = total_loss / max(total_valid_labels, 1)

    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    per_class_auroc = {}
    valid_aurocs = []

    for i, col in enumerate(TARGET_COLUMNS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_pred = all_probs[col_mask, i]

        if len(np.unique(y_true)) == 2:
            score = roc_auc_score(y_true, y_pred)
            per_class_auroc[col] = float(score)
            valid_aurocs.append(score)
        else:
            per_class_auroc[col] = None

    macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None
    return avg_loss, macro_auroc, per_class_auroc

In [ ]:
import os
from pathlib import Path
from io import BytesIO
from PIL import Image
import pandas as pd
from tqdm.auto import tqdm
import redivis

# 1. Ensure target cache directory exists
CACHE_ROOT_256 = Path("/content/dp_mmfl_image_cache_256/train")
CACHE_ROOT_256.mkdir(parents=True, exist_ok=True)

# 2. Select the identical deterministic 1,000 samples used across E1
manifest = pd.read_parquet(MANIFEST_PATH)
cache_manifest = (
    manifest[(manifest["experiment_split"] == "train") & (manifest["frontal_lateral"] == "Frontal")]
    .sort_values("sample_id")
    .head(1000)
    .reset_index(drop=True)
)

existing_256 = list(CACHE_ROOT_256.glob("*.png"))
print(f"Existing 256x256 images: {len(existing_256)}")

if len(existing_256) < 1000:
    print("Connecting to Redivis to stream and resize 1,000 images directly to 256x256...")
    table = redivis.table("aimi.chexpert_plus:5yyj:v1_0.png_train:s6cj")

    for row in tqdm(cache_manifest.itertuples(), total=len(cache_manifest), desc="Caching 256x256 images"):
        dst_path = CACHE_ROOT_256 / f"{int(row.sample_id):07d}.png"
        if dst_path.exists():
            continue

        png_path = str(row.path_to_image)
        if png_path.startswith("train/"):
            png_path = png_path[len("train/"):]
        if png_path.endswith(".jpg"):
            png_path = png_path[:-4] + ".png"

        file = table.file(png_path)
        data = file.read(as_text=False)
        img = Image.open(BytesIO(data)).convert("L")
        img = img.resize((256, 256), Image.Resampling.BILINEAR)
        img.save(dst_path, format="PNG")

total_ready = len(list(CACHE_ROOT_256.glob("*.png")))
print(f"\nCache ready: {total_ready}/1000 images at {CACHE_ROOT_256}")
assert total_ready == 1000, f"Expected 1,000 images, got {total_ready}"

Existing 256x256 images: 0
Connecting to Redivis to stream and resize 1,000 images directly to 256x256...


Caching 256x256 images:   0%|          | 0/1000 [00:00<?, ?it/s]

Please visit the URL below to authenticate with your Redivis account:
https://redivis.com/oauth/authorize?user_code=6f334269e50db5a1d304e31bc2629cae

Cache ready: 1000/1000 images at /content/dp_mmfl_image_cache_256/train


In [ ]:
import json
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
MANIFEST_PATH = PROJECT_ROOT / "data/processed/chexpert_plus_manifest.parquet"

SPLIT_PATH = (
    PROJECT_ROOT
    / "results/E1_image_baseline/metrics/split_info.json"
)

with open(SPLIT_PATH, "r") as f:
    split_info = json.load(f)

print("Split configuration keys:", list(split_info.keys()))
print(json.dumps(split_info, indent=2)[:4000])

Split configuration keys: ['train_sample_ids', 'val_sample_ids', 'train_patients', 'val_patients']
{
  "train_sample_ids": [
    0,
    4,
    6,
    8,
    12,
    13,
    14,
    16,
    17,
    18,
    28,
    30,
    31,
    32,
    33,
    38,
    40,
    41,
    42,
    44,
    45,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    59,
    60,
    67,
    69,
    73,
    74,
    75,
    77,
    78,
    79,
    80,
    81,
    82,
    83,
    84,
    85,
    86,
    87,
    88,
    89,
    90,
    91,
    92,
    93,
    94,
    95,
    96,
    97,
    98,
    101,
    102,
    103,
    104,
    105,
    106,
    107,
    108,
    110,
    111,
    112,
    113,
    114,
    115,
    116,
    117,
    118,
    119,
    120,
    121,
    122,
    123,
    125,
    129,
    130,
    132,
    133,
    134,
    135,
    136,
    137,
    139,
    140,
    142,
    143,
    144,
    146,
    147,
    148,
    150,
    151,
    152,
    153,
    

In [ ]:
manifest = pd.read_parquet(MANIFEST_PATH)

print("Manifest shape:", manifest.shape)
print("Has report_clean:", "report_clean" in manifest.columns)
print("Has report:", "report" in manifest.columns)
print("Has sample_id:", "sample_id" in manifest.columns)
print("Has patient ID:", "deid_patient_id" in manifest.columns)

print("\nRequired fields:")
for col in [
    "sample_id",
    "deid_patient_id",
    "report_clean",
    "frontal_lateral",
    "experiment_split",
]:
    print(f"{col}: {col in manifest.columns}")

if "report_clean" in manifest.columns:
    print("\nExample cleaned report:")
    print(str(manifest["report_clean"].dropna().iloc[0])[:1000])

Manifest shape: (223462, 58)
Has report_clean: True
Has report: True
Has sample_id: True
Has patient ID: True

Required fields:
sample_id: True
deid_patient_id: True
report_clean: True
frontal_lateral: True
experiment_split: True

Example cleaned report:
NARRATIVE: CHEST, ONE VIEW: 2-10-2001 FINDINGS: Costophrenic angles sharp, without evidence of effusion. The cardiomediastinal silhouette is normal. Vessels mildly indistinct with prominence of interstitial structures, suggesting mild, pulmonary edema. Left subclavian central venous catheter is seen, tip in mid SVC. No pneumothorax. IMPRESSION: 1. NO EVIDENCE OF PNEUMOTHORAX. 2. MILD INTERSTITIAL PULMONARY EDEMA. END OF IMPRESSION: SUMMARY: 2 I have personally reviewed the images for this examination and agree with the report transcribed above. By: Dr. Juarez Tucker N on: 2-10-2001 __________________________________


In [ ]:
import json
from pathlib import Path

SPLIT_PATH = (
    PROJECT_ROOT
    / "results/E1_image_baseline/metrics/split_info.json"
)

with open(SPLIT_PATH, "r") as f:
    split_info = json.load(f)

train_sample_ids = [int(x) for x in split_info["train_sample_ids"]]
val_sample_ids = [int(x) for x in split_info["val_sample_ids"]]

print("Train samples:", len(train_sample_ids))
print("Validation samples:", len(val_sample_ids))
print("Unique samples:", len(set(train_sample_ids + val_sample_ids)))
print("Train/validation overlap:", len(set(train_sample_ids) & set(val_sample_ids)))

assert len(train_sample_ids) == 835
assert len(val_sample_ids) == 165
assert len(set(train_sample_ids + val_sample_ids)) == 1000
assert not (set(train_sample_ids) & set(val_sample_ids))

Train samples: 835
Validation samples: 165
Unique samples: 1000
Train/validation overlap: 0


In [ ]:
import pandas as pd

manifest = pd.read_parquet(MANIFEST_PATH)

# Keep only frontal images represented by the E1 split.
all_sample_ids = train_sample_ids + val_sample_ids

subset = manifest[
    manifest["sample_id"].isin(all_sample_ids)
    & (manifest["frontal_lateral"] == "Frontal")
].copy()

assert subset["sample_id"].is_unique

missing_reports = subset["report_clean"].isna().sum()
empty_reports = (
    subset["report_clean"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Matched samples:", len(subset))
print("Missing reports:", missing_reports)
print("Empty reports:", empty_reports)

assert len(subset) == 1000
assert missing_reports == 0
assert empty_reports == 0

Matched samples: 1000
Missing reports: 0
Empty reports: 0


In [ ]:
E2_DIR = PROJECT_ROOT / "results" / "E2_multimodal"
E2_DIR.mkdir(parents=True, exist_ok=True)

E2_SPLIT_PATH = E2_DIR / "split_info.json"

with open(E2_SPLIT_PATH, "w") as f:
    json.dump(
        {
            "train_sample_ids": train_sample_ids,
            "val_sample_ids": val_sample_ids,
            "source_split": str(SPLIT_PATH),
            "subset_size": len(all_sample_ids),
            "text_column": "report_clean",
            "seed": 42,
        },
        f,
        indent=2,
    )

print("Saved E2 split:", E2_SPLIT_PATH)

Saved E2 split: /content/drive/MyDrive/DP-MMFL/results/E2_multimodal/split_info.json


In [ ]:
import json
import time
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
MAX_LENGTH = 384
TEXT_BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Keep a deterministic ordering for all embeddings.
all_sample_ids = train_sample_ids + val_sample_ids

text_manifest = (
    manifest[manifest["sample_id"].isin(all_sample_ids)]
    [["sample_id", "report_clean"]]
    .copy()
)

text_manifest["sample_id"] = text_manifest["sample_id"].astype(int)
text_manifest = (
    text_manifest
    .set_index("sample_id")
    .loc[all_sample_ids]
    .reset_index()
)

assert text_manifest["sample_id"].tolist() == all_sample_ids
assert text_manifest["report_clean"].notna().all()
assert text_manifest["report_clean"].str.strip().ne("").all()

print("Reports to encode:", len(text_manifest))
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
text_encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
text_encoder.eval()

for parameter in text_encoder.parameters():
    parameter.requires_grad = False

embedding_chunks = []
start = time.perf_counter()

with torch.inference_mode():
    for start_idx in range(0, len(text_manifest), TEXT_BATCH_SIZE):
        batch_reports = text_manifest["report_clean"].iloc[
            start_idx:start_idx + TEXT_BATCH_SIZE
        ].tolist()

        tokens = tokenizer(
            batch_reports,
            max_length=MAX_LENGTH,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )

        tokens = {
            key: value.to(device)
            for key, value in tokens.items()
        }

        outputs = text_encoder(**tokens)

        # CLS representation: one 768-dimensional vector per report.
        embeddings = outputs.last_hidden_state[:, 0, :]
        embedding_chunks.append(embeddings.cpu().float())

        if (start_idx // TEXT_BATCH_SIZE + 1) % 25 == 0:
            print(f"Encoded {min(start_idx + TEXT_BATCH_SIZE, len(text_manifest))}"
                  f"/{len(text_manifest)} reports")

text_embeddings = torch.cat(embedding_chunks, dim=0)

print("\nEmbedding shape:", tuple(text_embeddings.shape))
print("Embedding dtype:", text_embeddings.dtype)
print(f"Elapsed time: {time.perf_counter() - start:.1f} seconds")

assert text_embeddings.shape == (1000, 768)
assert torch.isfinite(text_embeddings).all()

embedding_path = (
    PROJECT_ROOT
    / "results/E2_multimodal/text_embeddings.pt"
)

torch.save(
    {
        "sample_ids": all_sample_ids,
        "embeddings": text_embeddings,
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "pooling": "CLS",
        "frozen_encoder": True,
    },
    embedding_path,
)

print("Saved embeddings:", embedding_path)

Reports to encode: 1000
Device: cuda


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  436MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoded 200/1000 reports
Encoded 400/1000 reports
Encoded 600/1000 reports
Encoded 800/1000 reports
Encoded 1000/1000 reports

Embedding shape: (1000, 768)
Embedding dtype: torch.float32
Elapsed time: 21.3 seconds
Saved embeddings: /content/drive/MyDrive/DP-MMFL/results/E2_multimodal/text_embeddings.pt


In [ ]:
import torch
from pathlib import Path

from dp_mmfl.models.image_model import ResNet18Classifier

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
EMBEDDING_PATH = PROJECT_ROOT / "results/E2_multimodal/text_embeddings.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

saved = torch.load(
    EMBEDDING_PATH,
    map_location="cpu",
    weights_only=False,
)

text_embeddings = saved["embeddings"]
embedding_sample_ids = [int(x) for x in saved["sample_ids"]]

assert text_embeddings.shape == (1000, 768)
assert embedding_sample_ids == train_sample_ids + val_sample_ids
assert torch.isfinite(text_embeddings).all()

image_model = ResNet18Classifier(
    num_classes=13,
    pretrained=True,
).to(device)

print("Text embeddings:", tuple(text_embeddings.shape))
print("Embedding IDs match E2 split:", True)
print("Image model device:", next(image_model.parameters()).device)
print("Image model output classes:", image_model.backbone.fc.out_features)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 98.3MB/s]


Text embeddings: (1000, 768)
Embedding IDs match E2 split: True
Image model device: cuda:0
Image model output classes: 13


In [ ]:
import torch
import torch.nn as nn
from torchvision.models import ResNet18_Weights, resnet18


class MultimodalClassifier(nn.Module):
    def __init__(self, num_classes=13, pretrained=True):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        self.image_encoder = resnet18(weights=weights)

        image_features = self.image_encoder.fc.in_features
        self.image_encoder.fc = nn.Identity()

        self.image_projection = nn.Sequential(
            nn.Linear(image_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.text_projection = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes),
        )

    def forward(self, images, text_embeddings):
        image_features = self.image_encoder(images)
        image_features = self.image_projection(image_features)

        text_features = self.text_projection(text_embeddings)

        fused_features = torch.cat(
            [image_features, text_features],
            dim=1,
        )

        return self.classifier(fused_features)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultimodalClassifier(
    num_classes=13,
    pretrained=True,
).to(device)

dummy_images = torch.randn(4, 3, 224, 224, device=device)
dummy_text = torch.randn(4, 768, device=device)

with torch.no_grad():
    dummy_logits = model(dummy_images, dummy_text)

print("Image batch:", tuple(dummy_images.shape))
print("Text batch:", tuple(dummy_text.shape))
print("Output logits:", tuple(dummy_logits.shape))

assert dummy_logits.shape == (4, 13)
assert torch.isfinite(dummy_logits).all()

print("Multimodal forward pass: PASS")

Image batch: (4, 3, 224, 224)
Text batch: (4, 768)
Output logits: (4, 13)
Multimodal forward pass: PASS


In [ ]:
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score

from dp_mmfl.evaluation.losses import masked_bce_with_logits
from dp_mmfl.data.labels import TARGET_COLUMNS

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
E2_DIR = PROJECT_ROOT / "results/E2_multimodal"
CHECKPOINT_DIR = E2_DIR / "checkpoints"
METRICS_DIR = E2_DIR / "metrics"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# Match embeddings by sample_id, never by shuffled row position.
embedding_by_id = {
    int(sample_id): text_embeddings[i]
    for i, sample_id in enumerate(embedding_sample_ids)
}

assert len(embedding_by_id) == 1000
assert set(embedding_by_id) == set(train_sample_ids + val_sample_ids)

print("Embedding lookup size:", len(embedding_by_id))
print("Train IDs aligned:", all(x in embedding_by_id for x in train_sample_ids))
print("Validation IDs aligned:", all(x in embedding_by_id for x in val_sample_ids))

Embedding lookup size: 1000
Train IDs aligned: True
Validation IDs aligned: True


In [ ]:
def multimodal_batches(loader, embedding_by_id, device):
    for batch in loader:
        ids = batch["sample_id"]

        if torch.is_tensor(ids):
            ids = ids.tolist()
        elif isinstance(ids, np.ndarray):
            ids = ids.tolist()

        ids = [int(x) for x in ids]

        text_batch = torch.stack(
            [embedding_by_id[sample_id] for sample_id in ids]
        ).to(device, non_blocking=True)

        images = batch["image"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        masks = batch["label_mask"].to(device, non_blocking=True)

        yield images, text_batch, labels, masks

In [ ]:
import time
import json
import copy
import torch
import pandas as pd
from sklearn.metrics import roc_auc_score

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MultimodalClassifier(num_classes=13, pretrained=True).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

EPOCHS = 3
best_val_loss = float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()

    model.train()
    train_loss_sum = 0.0
    train_batches = 0

    for images, text, labels, masks in multimodal_batches(
        train_loader, embedding_by_id, device
    ):
        optimizer.zero_grad(set_to_none=True)

        logits = model(images, text)
        loss = masked_bce_with_logits(logits, labels, masks)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    train_loss = train_loss_sum / max(train_batches, 1)

    model.eval()
    val_loss_sum = 0.0
    val_batches = 0
    all_targets, all_masks, all_probs = [], [], []

    with torch.no_grad():
        for images, text, labels, masks in multimodal_batches(
            val_loader, embedding_by_id, device
        ):
            logits = model(images, text)
            loss = masked_bce_with_logits(logits, labels, masks)

            val_loss_sum += loss.item()
            val_batches += 1

            all_targets.append(labels.cpu())
            all_masks.append(masks.cpu())
            all_probs.append(torch.sigmoid(logits).cpu())

    val_loss = val_loss_sum / max(val_batches, 1)

    targets = torch.cat(all_targets).numpy()
    masks = torch.cat(all_masks).numpy()
    probs = torch.cat(all_probs).numpy()

    per_label_aucs = []
    for j in range(targets.shape[1]):
        valid = masks[:, j] > 0.5
        y = targets[valid, j]
        p = probs[valid, j]

        # AUROC is undefined if this validation label has one class only.
        if len(y) > 0 and len(np.unique(y)) == 2:
            per_label_aucs.append(roc_auc_score(y, p))

    macro_auc = (
        float(np.mean(per_label_aucs))
        if per_label_aucs else float("nan")
    )

    record = {
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_auroc": macro_auc,
        "labels_with_defined_auc": len(per_label_aucs),
        "epoch_seconds": time.perf_counter() - epoch_start,
    }
    history.append(record)

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "metrics": record,
        "target_columns": TARGET_COLUMNS,
        "train_sample_ids": train_sample_ids,
        "val_sample_ids": val_sample_ids,
        "text_encoder": "emilyalsentzer/Bio_ClinicalBERT",
        "text_embedding_method": "frozen CLS embedding",
        "seed": 42,
    }

    torch.save(checkpoint, CHECKPOINT_DIR / "last.pt")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(checkpoint, CHECKPOINT_DIR / "best.pt")
        print(f"  Saved best checkpoint (val loss {val_loss:.4f})")

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss {train_loss:.4f} | "
        f"Val loss {val_loss:.4f} | "
        f"Macro AUROC {macro_auc:.4f} | "
        f"{record['epoch_seconds']:.1f}s"
    )

history_df = pd.DataFrame(history)
history_df.to_csv(METRICS_DIR / "training_history.csv", index=False)

with open(METRICS_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2, allow_nan=True)

print("\nE2 training complete.")
print("Best checkpoint:", CHECKPOINT_DIR / "best.pt")
print("History:", METRICS_DIR / "training_history.csv")

NameError: name 'train_loader' is not defined

In [ ]:
# Step 13 — Train E2 Multimodal Baseline with Auto-Checkpointing & Metric Logging
import time
import json
import copy

torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MultimodalClassifier(num_classes=13, pretrained=True).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

EPOCHS = 3
best_val_loss = float("inf")
history = []

print(f"Starting E2 Multimodal Training: {EPOCHS} epochs on {device.type.upper()}\n")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.perf_counter()

    # --- Training Phase ---
    model.train()
    train_loss_accum = 0.0
    train_valid_labels = 0

    for images, text_batch, labels, masks in multimodal_batches(train_loader, embedding_by_id, device):
        optimizer.zero_grad()
        logits = model(images, text_batch)
        loss = masked_bce_with_logits(logits, labels, masks)
        loss.backward()
        optimizer.step()

        batch_valid = masks.sum().item()
        train_loss_accum += loss.item() * batch_valid
        train_valid_labels += batch_valid

    avg_train_loss = train_loss_accum / max(train_valid_labels, 1)

    # --- Validation Phase ---
    model.eval()
    val_loss_accum = 0.0
    val_valid_labels = 0

    all_logits = []
    all_targets = []
    all_masks = []

    with torch.no_grad():
        for images, text_batch, labels, masks in multimodal_batches(val_loader, embedding_by_id, device):
            logits = model(images, text_batch)
            loss = masked_bce_with_logits(logits, labels, masks)

            batch_valid = masks.sum().item()
            val_loss_accum += loss.item() * batch_valid
            val_valid_labels += batch_valid

            all_logits.append(logits.cpu())
            all_targets.append(labels.cpu())
            all_masks.append(masks.cpu())

    avg_val_loss = val_loss_accum / max(val_valid_labels, 1)

    # Compute validation AUROC per pathology
    all_probs = torch.sigmoid(torch.cat(all_logits, dim=0)).numpy()
    all_targets = torch.cat(all_targets, dim=0).numpy()
    all_masks = torch.cat(all_masks, dim=0).numpy()

    per_class_auroc = {}
    valid_aurocs = []

    for i, col in enumerate(TARGET_COLUMNS):
        col_mask = all_masks[:, i] == 1.0
        y_true = all_targets[col_mask, i]
        y_pred = all_probs[col_mask, i]

        if len(np.unique(y_true)) == 2:
            score = roc_auc_score(y_true, y_pred)
            per_class_auroc[col] = float(score)
            valid_aurocs.append(score)
        else:
            per_class_auroc[col] = None

    val_macro_auroc = float(np.mean(valid_aurocs)) if valid_aurocs else None
    epoch_duration = time.perf_counter() - epoch_start

    auroc_str = f"{val_macro_auroc:.4f}" if val_macro_auroc is not None else "N/A"
    print(
        f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_duration:.1f}s] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Macro AUROC: {auroc_str}"
    )

    epoch_record = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "val_macro_auroc": val_macro_auroc,
        "per_class_auroc": per_class_auroc,
        "duration_sec": epoch_duration,
    }
    history.append(epoch_record)

    # Per-epoch checkpoint
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": avg_train_loss,
            "val_loss": avg_val_loss,
            "val_macro_auroc": val_macro_auroc,
        },
        CHECKPOINT_DIR / f"epoch_{epoch:02d}.pt",
    )

    # Save best checkpoint
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), CHECKPOINT_DIR / "best_model.pt")
        print(f"  --> Saved new best multimodal model (Val Loss: {avg_val_loss:.4f})")

# Save complete training history
with open(METRICS_DIR / "e2_training_history.json", "w") as f:
    json.dump(history, f, indent=2)

print("\nE2 multimodal training complete. All checkpoints and metrics saved to Drive.")

Starting E2 Multimodal Training: 3 epochs on CUDA

Epoch 01/03 [4.9s] | Train Loss: 0.5651 | Val Loss: 0.4790 | Val Macro AUROC: 0.4672
  --> Saved new best multimodal model (Val Loss: 0.4790)
Epoch 02/03 [3.8s] | Train Loss: 0.4351 | Val Loss: 0.4720 | Val Macro AUROC: 0.4738
  --> Saved new best multimodal model (Val Loss: 0.4720)
Epoch 03/03 [3.8s] | Train Loss: 0.4059 | Val Loss: 0.4808 | Val Macro AUROC: 0.5209

E2 multimodal training complete. All checkpoints and metrics saved to Drive.


In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader
from dp_mmfl.data.preprocessed_cached_dataset import PreprocessedCachedImageDataset

# 1. Image transforms
train_transform = transforms.Compose([
    transforms.RandomCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

eval_transform = transforms.Compose([
    transforms.CenterCrop(224),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# 2. Instantiate datasets for train and val splits
train_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=train_sample_ids,
    image_transform=train_transform,
)

val_dataset = PreprocessedCachedImageDataset(
    manifest_path=MANIFEST_PATH,
    cache_root=CACHE_ROOT_256,
    sample_ids=val_sample_ids,
    image_transform=eval_transform,
)

# 3. Create loaders
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=True,
)

print(f"train_loader: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"val_loader:   {len(val_dataset)} samples ({len(val_loader)} batches)")

train_loader: 835 samples (27 batches)
val_loader:   165 samples (6 batches)


In [ ]:
import os
import json
import shutil
from pathlib import Path
import pandas as pd
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/DP-MMFL")
E2_DIR = PROJECT_ROOT / "results" / "E2_multimodal"
CHECKPOINT_DIR = E2_DIR / "checkpoints"
METRICS_DIR = E2_DIR / "metrics"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

# 1. Standardize Checkpoint Names (best.pt and last.pt)
best_model_path = CHECKPOINT_DIR / "best_model.pt"
best_target_path = CHECKPOINT_DIR / "best.pt"
if best_model_path.exists():
    shutil.copy2(best_model_path, best_target_path)

last_epoch_path = CHECKPOINT_DIR / "epoch_03.pt"
last_target_path = CHECKPOINT_DIR / "last.pt"
if last_epoch_path.exists():
    shutil.copy2(last_epoch_path, last_target_path)

# 2. Export Metrics to CSV and JSON
history_json_path = METRICS_DIR / "e2_training_history.json"
metrics_csv_path = METRICS_DIR / "training_history.csv"
metrics_json_path = METRICS_DIR / "training_history.json"

if history_json_path.exists():
    with open(history_json_path, "r") as f:
        history_data = json.load(f)

    # Save standard training_history.json
    with open(metrics_json_path, "w") as f:
        json.dump(history_data, f, indent=2)

    # Flatten training_history.csv for easy plotting and reporting
    flat_rows = []
    for row in history_data:
        r = {
            "epoch": row["epoch"],
            "train_loss": row["train_loss"],
            "val_loss": row["val_loss"],
            "val_macro_auroc": row["val_macro_auroc"],
            "duration_sec": row["duration_sec"],
        }
        if "per_class_auroc" in row and row["per_class_auroc"]:
            for k, v in row["per_class_auroc"].items():
                r[f"auroc_{k}"] = v
        flat_rows.append(r)
    pd.DataFrame(flat_rows).to_csv(metrics_csv_path, index=False)

# 3. Persist Text Embeddings & Split Information
text_embeddings_target = E2_DIR / "text_embeddings.pt"
if "text_embeddings" in globals() and not text_embeddings_target.exists():
    torch.save(
        {
            "text_embeddings": text_embeddings,
            "embedding_sample_ids": embedding_sample_ids,
        },
        text_embeddings_target,
    )

split_target = E2_DIR / "split_info.json"
split_source = PROJECT_ROOT / "results" / "E1_image_baseline" / "metrics" / "split_info.json"
if split_source.exists() and not split_target.exists():
    shutil.copy2(split_source, split_target)

# 4. Verification Check
expected_files = [
    CHECKPOINT_DIR / "best.pt",
    CHECKPOINT_DIR / "last.pt",
    METRICS_DIR / "training_history.csv",
    METRICS_DIR / "training_history.json",
    E2_DIR / "text_embeddings.pt",
    E2_DIR / "split_info.json",
]

print("=== E2 Artifacts Verification ===")
all_present = True
for p in expected_files:
    exists = p.exists()
    status = "OK" if exists else "MISSING"
    size_str = f"({p.stat().st_size / 1024:.1f} KB)" if exists else ""
    print(f"[{status:7s}] {p.relative_to(PROJECT_ROOT)} {size_str}")
    if not exists:
        all_present = False

print("\nAll E2 artifacts preserved and verified:", all_present)

=== E2 Artifacts Verification ===
[OK     ] results/E2_multimodal/checkpoints/best.pt (45549.0 KB)
[OK     ] results/E2_multimodal/checkpoints/last.pt (136537.9 KB)
[OK     ] results/E2_multimodal/metrics/training_history.csv (1.1 KB)
[OK     ] results/E2_multimodal/metrics/training_history.json (2.1 KB)
[OK     ] results/E2_multimodal/text_embeddings.pt (3004.5 KB)
[OK     ] results/E2_multimodal/split_info.json (9.2 KB)

All E2 artifacts preserved and verified: True
